In [1]:
import sys, os

dir = os.path.abspath('..')
sys.path.append(os.path.join(dir))
while not os.path.basename(dir) == 'v2':
    parent = os.path.dirname(dir)
    if parent == dir:
        raise FileNotFoundError("No parent directory named 'v2' found.")
    dir = parent

In [2]:
DATA_NAME = 'jorge-give-and-go'

In [3]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{DATA_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=.5)

print(f'Number of frames: {[len(d.video.frame_dir) for d in demos]}')

Importing:   0%|          | 0/3 [00:00<?, ?it/s]

FPS: 5.0
Samplerate: 0.5
Total # of frames: 327
Duration of Video: 65.4 seconds


Importing:  33%|███▎      | 1/3 [00:07<00:14,  7.02s/it]

Total # of frames saved: 164
FPS: 5.0
Samplerate: 0.5
Total # of frames: 243
Duration of Video: 48.6 seconds


Importing:  67%|██████▋   | 2/3 [00:12<00:05,  5.91s/it]

Total # of frames saved: 122
FPS: 5.0
Samplerate: 0.5
Total # of frames: 199
Duration of Video: 39.8 seconds


Importing: 100%|██████████| 3/3 [00:16<00:00,  5.45s/it]

Total # of frames saved: 100
Number of frames: [164, 122, 100]


In [11]:
DATA_DIR

'/Users/wiktorrajca/Documents/GitHub/narrated_demo/v2/data/jorge-give-and-go'

In [4]:
from scenic_fc.api import api
from infer_utils import Inference
acts = Inference.act_from(demos, api, model='gpt-4o')

100%|██████████| 3/3 [00:17<00:00,  5.89s/it]

Inference done.


In [5]:
from scenic_fc.api import api
from infer_utils_gemini import Inference_Gemini
acts_gemini = Inference_Gemini.act_from(demos, api)

Running Gemini inference: 100%|██████████| 3/3 [00:55<00:00, 18.41s/it]

Inference done.


In [6]:
combined = Inference.combine(acts=acts, api=api, model='gpt-4o')

Editing done.


In [7]:
combined_gemini = Inference_Gemini.combine(acts=acts_gemini, api=api)

Editing done.


In [8]:
import copy
final = copy.deepcopy(combined)
final_gemini = copy.deepcopy(combined_gemini)

In [9]:
%load_ext autoreload
%autoreload 2

from edit_utils import edit_interface, EditLogger

logger = EditLogger()
edit_interface(final, api, logger)

In [10]:
%load_ext autoreload
%autoreload 2

from edit_utils import edit_interface, EditLogger

logger = EditLogger()
edit_interface(final_gemini, api, logger)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import json

infer_data = combined.to_dict(expanded=False)
with open(f'exports/{DATA_NAME}_infer.json', 'w') as f:
        json.dump(infer_data, f, indent=4)

edit_data = final.to_dict(expanded=False)
with open(f'exports/{DATA_NAME}_edit.json', 'w') as f:
        json.dump(edit_data, f, indent=4)

In [15]:
%load_ext autoreload
%autoreload 2

from synth_utils import Synth

synth = Synth(final, demos, api)
synth.run(logger)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Synthesizing:   0%|          | 0/9 [00:00<?, ?it/s]

distance from Coach to opponent
DistanceTo {'to': 'opponent', 'from': 'Coach', 'operator': 'greater_than'}
HorizontalRelation {'obj': 'Coach', 'relation': 'right', 'ref': 'teammate'}


Synthesizing:  11%|█         | 1/9 [00:23<03:11, 23.90s/it]

distance from Coach to opponent
distance from Coach to goal
DistanceTo {'to': 'opponent', 'from': 'Coach', 'operator': 'greater_than'}
DistanceTo {'to': 'goal', 'from': 'Coach', 'operator': 'less_than'}


...

HasBallPossession {'player': 'Coach'}


Synthesizing:  89%|████████▉ | 8/9 [01:25<00:08,  8.40s/it]

MovingTowards {'obj': 'teammate', 'ref': 'goal'}


Synthesizing: 100%|██████████| 9/9 [01:49<00:00, 12.19s/it]

Pressure {'player1': 'opponent', 'player2': 'Coach'}
HasPath {'obj1': 'Coach', 'obj2': 'goal'}


In [23]:
synth_data = synth.to_dict(expanded=True)
with open(f'exports/{DATA_NAME}_synth.json', 'w') as f:
        json.dump(synth_data, f, indent=4)